# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
# The file uses an older text encoding, so I use latin-1 to avoid read errors
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Summary statistics for the numeric columns
df.describe()

In [ ]:
# How many values are missing in each column? (biggest first)
df.isna().sum().sort_values(ascending=False)

**What I notice**
- The dataset has about 88,889 rows and 31 columns.
- `Aircraft.Category` is missing in over 56,000 rows. I need to deal with that carefully because it is how I find airplanes.
- The injury columns have missing values too.
- The column names have dots and capital letters (like `Total.Fatal.Injuries`). To make them easier to type I will rename them to lowercase with underscores (e.g. `total_fatal_injuries`).

In [ ]:
df.columns = df.columns.str.replace('.', '_', regex=False).str.lower()
df.columns.tolist()

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
# Look at the columns I will use to filter
print(df['investigation_type'].value_counts(), '\n')
print(df['amateur_built'].value_counts(dropna=False), '\n')
print(df['aircraft_category'].value_counts(dropna=False))

# Convert the date and make a year column
df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
df['year'] = df['event_date'].dt.year

# How much of aircraft_category is missing in each 5-year period?
df.groupby(df['year'] // 5 * 5)['aircraft_category'].apply(lambda s: s.isna().mean()).round(2)

**Imputing the missing `aircraft_category`**

Almost all accidents before 2005 have no category. If I dropped them I would lose most of the data the client cares about (1983 onwards). 

*My idea:* if a make/model is labelled 'Airplane' in other rows, then the blank rows for the same make/model are very likely airplanes too. I take the most common category for each make/model and use it to fill the gaps. Rows that are still unknown after this are dropped, because I can't be sure they are airplanes.

In [ ]:
# temporary key to match the same plane type (uppercase, no extra spaces)
df['temp_key'] = df['make'].str.upper().str.strip() + ' ' + df['model'].str.upper().str.strip()

known = df.dropna(subset=['aircraft_category'])
category_lookup = known.groupby('temp_key')['aircraft_category'].agg(lambda s: s.mode().iloc[0])

print('Missing category before:', df['aircraft_category'].isna().sum())
df['aircraft_category'] = df['aircraft_category'].fillna(df['temp_key'].map(category_lookup))
print('Missing category after: ', df['aircraft_category'].isna().sum())
df = df.drop(columns='temp_key')

**Filtering** - the client only wants:
- accidents from **1983 onwards** (a plane model is assumed to retire after 40 years; 2023 - 40 = 1983)
- **accidents** (not 'incidents')
- **professional builds** (`amateur_built == 'No'`)
- **airplanes** (the data also has helicopters, gliders, balloons...)

In [ ]:
print('Start:', len(df))
df = df[df['year'] >= 1983];                    print('1983 onwards:', len(df))
df = df[df['investigation_type'] == 'Accident']; print('accidents only:', len(df))
df = df[df['amateur_built'] == 'No'];            print('professional builds:', len(df))
df = df[df['aircraft_category'] == 'Airplane'].copy(); print('airplanes:', len(df))

# check we kept data from every time period (not just recent years)
df.groupby(df['year'] // 5 * 5).size()

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
injury_cols = ['total_fatal_injuries', 'total_serious_injuries', 'total_minor_injuries', 'total_uninjured']

# Assumption: a missing injury count means 0 people in that category
df[injury_cols] = df[injury_cols].fillna(0)

# Estimated number of people on the plane = fatal + serious + minor + uninjured
df['total_passengers'] = df[injury_cols].sum(axis=1)

# We can't calculate a fraction if nobody was aboard, so drop those rows
df = df[df['total_passengers'] > 0].copy()

# Derived columns
df['fatal_or_serious'] = df['total_fatal_injuries'] + df['total_serious_injuries']
df['fatal_serious_share'] = df['fatal_or_serious'] / df['total_passengers']   # fraction of people aboard fatally/seriously hurt
df['severe_injury'] = (df['fatal_or_serious'] > 0).astype(int)                # 1 if anyone was fatally/seriously hurt

print('Rows left:', len(df))
df[['total_passengers', 'fatal_or_serious', 'fatal_serious_share', 'severe_injury']].describe()

**Assumptions and new columns**
- `total_passengers` = fatal + serious + minor + uninjured. It is an *estimate* of everyone aboard (crew included), since the data has no direct passenger count.
- Missing injury counts were filled with 0. This is an assumption: the NTSB probably leaves a count blank when there were none, but it may hide some missing data.
- `fatal_serious_share` = (fatal + serious) / total_passengers. This is the likelihood of a serious/fatal injury, as a number between 0 and 1.
- `severe_injury` = 1 if at least one person was fatally/seriously injured (a simple yes/no version).

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
print(df['aircraft_damage'].value_counts(dropna=False))

# Drop rows where we don't know the damage level
df = df.dropna(subset=['aircraft_damage'])
df = df[df['aircraft_damage'] != 'Unknown'].copy()

# 1 if the aircraft was destroyed, 0 otherwise
df['destroyed'] = (df['aircraft_damage'] == 'Destroyed').astype(int)

print('\nDestroyed rate overall:', round(df['destroyed'].mean(), 3))
df['aircraft_damage'].value_counts()

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning tasks for `make`:**
1. Drop rows with a missing make.
2. Make the text uppercase and remove extra spaces (so `Cessna` and `CESSNA ` match).
3. Fix a few manufacturers written in different ways (I found these with `value_counts`).
4. Keep only makes with **at least 50 accidents** so averages are based on enough data.

In [ ]:
df = df.dropna(subset=['make']).copy()
df['make'] = df['make'].str.upper().str.strip()

# Same manufacturer written differently
make_fixes = {'THE BOEING COMPANY': 'BOEING', 'CESSNA AIRCRAFT COMPANY': 'CESSNA',
              'AIRBUS INDUSTRIE': 'AIRBUS', 'BOMBARDIER INC': 'BOMBARDIER',
              'DOUGLAS': 'MCDONNELL DOUGLAS', 'DEHAVILLAND': 'DE HAVILLAND',
              'AVIAT': 'AVIAT AIRCRAFT INC', 'GRUMMAN ACFT ENG COR-SCHWEIZER': 'GRUMMAN-SCHWEIZER'}
df['make'] = df['make'].replace(make_fixes)

make_counts = df['make'].value_counts()
print('Number of makes before filtering:', len(make_counts))

keep_makes = make_counts[make_counts >= 50].index
df = df[df['make'].isin(keep_makes)].copy()

print('Number of makes kept (50+ accidents):', len(keep_makes))
print('Rows left:', len(df))
df['make'].value_counts().head(15)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
df = df.dropna(subset=['model']).copy()
df['model'] = df['model'].str.upper().str.strip()
df['model'] = df['model'].str.replace(r'^B(7\d\d)', r'\1', regex=True)   # 'B737' -> '737'

# Are model names unique to each make? (count how many makes use the same model name)
makes_per_model = df.groupby('model')['make'].nunique()
print('Model names used by more than one make:', (makes_per_model > 1).sum())
print(makes_per_model.sort_values(ascending=False).head(5))

**Result:** model names are *not* unique to a make (e.g. the same model name can appear under more than one make). So I create a unique identifier by joining make and model.

I make two identifiers:
- `make_model`: the exact model, e.g. `BOEING 737-300`
- `plane_type`: the *family*, e.g. `BOEING 737-300` and `BOEING 737-800` both become `BOEING 737`. Many exact models have only a few accidents, so grouping into families gives bigger samples and more reliable comparisons.

In [ ]:
df['make_model'] = df['make'] + ' ' + df['model']

# family = first part of the model name: '737-300' -> '737', '172S' -> '172', 'PA-28-140' -> 'PA-28'
family = df['model'].str.extract(r'^([A-Z]*-?\d+)')[0].fillna(df['model'])
family = family.str.replace('-', '', regex=False)        # 'MD-11' and 'MD11' become the same
df['plane_type'] = df['make'] + ' ' + family

print('Unique make_model:', df['make_model'].nunique())
print('Unique plane_type:', df['plane_type'].nunique())
df[['make', 'model', 'make_model', 'plane_type']].sample(8, random_state=1)

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# Weather: replace 'UNK' with NaN
print(df['weather_condition'].value_counts(dropna=False), '\n')
df['weather_condition'] = df['weather_condition'].str.upper().replace({'UNK': np.nan, 'UNKNOWN': np.nan})

# Engine type: 'Unknown' -> NaN
print(df['engine_type'].value_counts(dropna=False), '\n')
df['engine_type'] = df['engine_type'].replace({'Unknown': np.nan})

# Number of engines: check for strange values (0 engines = probably a glider/error)
print(df['number_of_engines'].value_counts(dropna=False), '\n')

# Purpose of flight and phase of flight: uppercase, unknown/other -> NaN
print(df['purpose_of_flight'].value_counts(dropna=False).head(10), '\n')
df['purpose_of_flight'] = df['purpose_of_flight'].str.upper().replace({'UNKNOWN': np.nan, 'OTHER WORK USE': 'OTHER'})

print(df['broad_phase_of_flight'].value_counts(dropna=False).head(15))
df['broad_phase_of_flight'] = df['broad_phase_of_flight'].str.upper().replace({'UNKNOWN': np.nan, 'OTHER': np.nan})

**Cleaning done for these columns:** uppercase text, 'Unknown'/'UNK' turned into NaN (so unknown is not counted as its own category), and I left the other NaNs in place. Planes with 0 engines will be removed later in the analysis if I use the engine count.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Part 121 means an airline (large commercial) flight. I use this later to tell large from small planes,
# so I create the column BEFORE dropping far_description (it has a lot of NaNs).
df['part121'] = df['far_description'].str.contains('121', na=False)

# share of NaNs in each column
na_share = df.isna().mean().sort_values(ascending=False)
print(na_share.head(12).round(2))

# drop columns with more than 50% missing values
too_many_nans = na_share[na_share > 0.5].index.tolist()
print('\nDropping:', too_many_nans)
df = df.drop(columns=too_many_nans)
df.shape

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [1754]:
df.to_csv('data/cleaned_aviation_data.csv', index=False)
print('Saved', df.shape)